# Speech Emotion Recognition

Add these 4 datasets before running (Add Data → search by name):
- `uwrfkaggler/ravdess-emotional-speech-audio`
- `ejlok1/toronto-emotional-speech-set-tess`
- `ejlok1/cremad`
- `ejlok1/surrey-audiovisual-expressed-emotion-savee`

Hit Run All. Fine to close the browser — Kaggle keeps running in the background.

In [ ]:
!pip install librosa -q

In [ ]:
import os, glob, pickle, warnings, gc
import numpy as np
import librosa
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
SAMPLE_RATE   = 22050
DURATION      = 3.0
N_SAMPLES     = int(SAMPLE_RATE * DURATION)
N_MFCC        = 40
N_FFT         = 2048
HOP_LENGTH    = 512
N_MELS        = 128
BATCH_SIZE    = 64
EPOCHS        = 80
PATIENCE      = 25
WARMUP_EPOCHS = 5
REG           = regularizers.l2(1e-4)
TARGET_TIME   = int(np.ceil(N_SAMPLES / HOP_LENGTH)) + 1
OUTPUT_DIR    = '/kaggle/working/'

VALID_EMOTIONS = {'neutral','calm','happy','sad','angry','fear','disgust','surprise'}

def _find_dataset(suffixes):
    """Auto-detect dataset path regardless of how Kaggle mounts it."""
    for suffix in suffixes:
        candidates = glob.glob(f'/kaggle/input/**/{suffix}', recursive=True)
        if candidates:
            return sorted(candidates, key=len)[-1]
    return None

RAVDESS_PATH = _find_dataset(['ravdess-emotional-speech-audio', 'ravdess']) or '/kaggle/input/ravdess-emotional-speech-audio'
TESS_PATH    = _find_dataset(['toronto-emotional-speech-set-tess', 'tess'])  or '/kaggle/input/toronto-emotional-speech-set-tess'
CREMAD_PATH  = _find_dataset(['cremad'])                                      or '/kaggle/input/cremad'
SAVEE_PATH   = _find_dataset(['surrey-audiovisual-expressed-emotion-savee', 'savee']) or '/kaggle/input/surrey-audiovisual-expressed-emotion-savee'

for name, path in [('RAVDESS',RAVDESS_PATH),('TESS',TESS_PATH),('CREMA-D',CREMAD_PATH),('SAVEE',SAVEE_PATH)]:
    status = 'OK' if os.path.exists(path) else 'MISSING — add via Add Data button'
    print(f'  {name}: {status} ({path})')

In [ ]:
def parse_ravdess(base):
    emap = {'01':'neutral','02':'calm','03':'happy','04':'sad',
            '05':'angry','06':'fear','07':'disgust','08':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        parts = os.path.basename(f).split('-')
        if len(parts) >= 3:
            e = emap.get(parts[2])
            if e in VALID_EMOTIONS: paths.append(f); labels.append(e)
    return paths, labels

def parse_tess(base):
    # TESS filenames: OAF_back_angry.wav, OAF_bar_neutral.wav, OAF_base_ps.wav (ps = pleasant surprise)
    emap = {'angry':'angry','disgust':'disgust','fear':'fear','happy':'happy',
            'neutral':'neutral','sad':'sad','ps':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        stem = os.path.splitext(os.path.basename(f))[0].lower()
        parts = stem.split('_')
        emotion_part = parts[-1] if parts else ''
        e = emap.get(emotion_part)
        if e: paths.append(f); labels.append(e)
    return paths, labels

def parse_cremad(base):
    emap = {'ANG':'angry','DIS':'disgust','FEA':'fear',
            'HAP':'happy','NEU':'neutral','SAD':'sad'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        parts = os.path.basename(f).split('_')
        if len(parts) >= 3:
            e = emap.get(parts[2])
            if e: paths.append(f); labels.append(e)
    return paths, labels

def parse_savee(base):
    emap = {'a':'angry','d':'disgust','f':'fear',
            'h':'happy','n':'neutral','sa':'sad','su':'surprise'}
    paths, labels = [], []
    for f in glob.glob(os.path.join(base,'**/*.wav'), recursive=True):
        stem   = os.path.splitext(os.path.basename(f))[0].lower()
        prefix = ''.join(c for c in stem if not c.isdigit())
        e = emap.get(prefix)
        if e: paths.append(f); labels.append(e)
    return paths, labels

all_paths, all_labels = [], []
for parser, path in [(parse_ravdess, RAVDESS_PATH), (parse_tess, TESS_PATH),
                     (parse_cremad,  CREMAD_PATH),  (parse_savee, SAVEE_PATH)]:
    if os.path.exists(path):
        p, l = parser(path)
        all_paths.extend(p); all_labels.extend(l)
        print(f'  {os.path.basename(path)}: {len(p)} files')

print(f'\nTotal: {len(all_paths)} files')
print(f'Distribution: {Counter(all_labels)}')

In [ ]:
def load_audio(fp):
    a, _ = librosa.load(fp, sr=SAMPLE_RATE, duration=DURATION)
    a, _ = librosa.effects.trim(a, top_db=25)
    return np.pad(a, (0, max(0, N_SAMPLES-len(a))))[:N_SAMPLES]

def extract_features(audio):
    feats = []
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for fn in [np.mean, np.std, np.min, np.max]:
        feats.extend(fn(mfcc, axis=1))
    for order in [1, 2]:
        d = librosa.feature.delta(mfcc, order=order)
        feats.extend(np.mean(d,axis=1)); feats.extend(np.std(d,axis=1))
    chroma = librosa.feature.chroma_stft(y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.extend(np.mean(chroma,axis=1)); feats.extend(np.std(chroma,axis=1))
    mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mdb = librosa.power_to_db(mel, ref=np.max)
    feats.extend(np.mean(mdb,axis=1)); feats.extend(np.std(mdb,axis=1))
    contrast = librosa.feature.spectral_contrast(y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.extend(np.mean(contrast,axis=1)); feats.extend(np.std(contrast,axis=1))
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(audio), sr=SAMPLE_RATE)
    feats.extend(np.mean(tonnetz,axis=1)); feats.extend(np.std(tonnetz,axis=1))
    zcr = librosa.feature.zero_crossing_rate(audio)
    feats.extend([np.mean(zcr), np.std(zcr), np.max(zcr)])
    rms = librosa.feature.rms(y=audio)
    feats.extend([np.mean(rms), np.std(rms), np.max(rms)])
    for fn in [librosa.feature.spectral_centroid, librosa.feature.spectral_bandwidth, librosa.feature.spectral_rolloff]:
        f = fn(y=audio, sr=SAMPLE_RATE)
        feats.extend([np.mean(f), np.std(f), np.max(f)])
    pitches, _ = librosa.piptrack(y=audio, sr=SAMPLE_RATE)
    pv = pitches[pitches > 0]
    feats.extend([np.mean(pv), np.std(pv), np.min(pv), np.max(pv)] if len(pv) > 0 else [0,0,0,0])
    return np.array(feats, dtype=np.float32)

def extract_mel_multiscale(audio):
    channels = []
    for nm, nf, hl in [(N_MELS, N_FFT, HOP_LENGTH),
                        (N_MELS, N_FFT//2, HOP_LENGTH//2),
                        (N_MELS, N_FFT*2,  HOP_LENGTH*2)]:
        mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=nm, n_fft=nf, hop_length=hl)
        mdb = librosa.power_to_db(mel, ref=np.max)
        mdb = (mdb - mdb.min()) / (mdb.max() - mdb.min() + 1e-8)
        mdb = np.pad(mdb,((0,0),(0,max(0,TARGET_TIME-mdb.shape[1]))))[:,:TARGET_TIME]
        channels.append(mdb)
    return np.stack(channels, axis=-1).astype(np.float32)

In [ ]:
def apply_rir(audio):
    rl  = int(SAMPLE_RATE * np.random.uniform(0.02, 0.2))
    rir = np.random.randn(rl) * np.exp(-np.linspace(0, 8, rl))
    rir /= (np.abs(rir).max() + 1e-8)
    return audio * 0.5 + np.convolve(audio, rir)[:len(audio)] * 0.5

def augment_audio(audio):
    choices = np.random.choice(['noise','pitch','stretch','shift','rir','gain'],
                                size=np.random.randint(1,3), replace=False)
    a = audio.copy()
    for aug in choices:
        if   aug=='noise':   a += np.random.randn(len(a)) * np.random.uniform(0.002,0.01)
        elif aug=='pitch':   a  = librosa.effects.pitch_shift(y=a, sr=SAMPLE_RATE, n_steps=np.random.uniform(-3,3))
        elif aug=='stretch': a  = librosa.effects.time_stretch(y=a, rate=np.random.uniform(0.8,1.2))
        elif aug=='shift':   a  = np.roll(a, int(np.random.uniform(-0.2,0.2)*len(a)))
        elif aug=='rir':     a  = apply_rir(a)
        elif aug=='gain':    a  = a * np.random.uniform(0.7,1.3)
    return np.pad(a[:N_SAMPLES],(0,max(0,N_SAMPLES-len(a))))

def spec_augment(mel3, fm=15, tm=25, n=2):
    m = mel3.copy(); h, w = m.shape[:2]
    for _ in range(n):
        f  = np.random.randint(1, fm+1); f0 = np.random.randint(0, max(1,h-f))
        m[f0:f0+f, :, :] = 0
        t  = np.random.randint(1, tm+1); t0 = np.random.randint(0, max(1,w-t))
        m[:, t0:t0+t, :] = 0
    return m

In [ ]:
print('Extracting features...')
X_flat_base, X_2d_base, y_base = [], [], []

for i, (path, label) in enumerate(zip(all_paths, all_labels)):
    try:
        audio = load_audio(path)
        X_flat_base.append(extract_features(audio))
        X_2d_base.append(extract_mel_multiscale(audio))
        y_base.append(label)
    except: pass
    if (i+1) % 1000 == 0: print(f'  {i+1}/{len(all_paths)}')

X_flat_base = np.array(X_flat_base, dtype=np.float32)
X_2d_base   = np.array(X_2d_base,   dtype=np.float32)
le          = LabelEncoder()
y_int       = le.fit_transform(y_base)
y_cat       = to_categorical(y_int)
NC          = len(le.classes_)
print(f'Classes ({NC}): {le.classes_}')
print(f'X_flat: {X_flat_base.shape}, X_2d: {X_2d_base.shape}')

In [ ]:
AUG_ROUNDS = 3

idx = np.arange(N_orig := len(y_base))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.15, stratify=y_int, random_state=42)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.5, stratify=y_int[idx_tmp], random_state=42)

X_va_f = X_flat_base[idx_va]; X_va_2 = X_2d_base[idx_va]; y_va = y_cat[idx_va]
X_te_f = X_flat_base[idx_te]; X_te_2 = X_2d_base[idx_te]; y_te = y_cat[idx_te]

scaler = StandardScaler().fit(X_flat_base[idx_tr])
X_va_f = scaler.transform(X_va_f)
X_te_f = scaler.transform(X_te_f)

y_tr_int  = y_int[idx_tr]
cw        = compute_class_weight('balanced', classes=np.unique(y_tr_int), y=y_tr_int)
class_wts = {i: cw[i] for i in range(len(cw))}

with open(OUTPUT_DIR+'label_encoder.pkl','wb') as f: pickle.dump(le, f)
with open(OUTPUT_DIR+'scaler.pkl','wb') as f:        pickle.dump(scaler, f)
print(f'Train: {len(idx_tr)}  Val: {len(idx_va)}  Test: {len(idx_te)}')

In [ ]:
class AugMixupGen(tf.keras.utils.Sequence):
    def __init__(self, X2, Xf, y, bs=64, alpha=0.4, aug_rounds=3, noise_std=0.02):
        self.X2 = X2; self.Xf = Xf; self.y = y
        self.bs = bs; self.alpha = alpha; self.noise_std = noise_std
        self.n = len(y)
        self.idx = np.arange(self.n * (aug_rounds + 1))
        np.random.shuffle(self.idx)

    def __len__(self): return int(np.ceil(len(self.idx) / self.bs))

    def on_epoch_end(self): np.random.shuffle(self.idx)

    def __getitem__(self, i):
        bi       = self.idx[i*self.bs:(i+1)*self.bs]
        orig_idx = bi % self.n
        do_aug   = bi >= self.n

        X2b = self.X2[orig_idx].copy()
        Xfb = self.Xf[orig_idx].copy()
        yb  = self.y[orig_idx]

        # Fast in-memory augmentation — no disk I/O, no librosa
        if do_aug.any():
            for j in np.where(do_aug)[0]:
                X2b[j] = spec_augment(X2b[j])
                X2b[j] *= np.random.uniform(0.85, 1.15)   # random gain
            n_aug = do_aug.sum()
            Xfb[do_aug] += (np.random.randn(n_aug, Xfb.shape[1]) * self.noise_std).astype(np.float32)

        # Mixup
        si  = np.random.permutation(len(bi))
        lam = np.maximum(l := np.random.beta(self.alpha, self.alpha, len(bi)), 1-l)
        l4  = lam.reshape(-1,1,1,1).astype(np.float32)
        l2  = lam.reshape(-1,1).astype(np.float32)
        return ({'spectrogram': l4*X2b + (1-l4)*X2b[si],
                 'features':    l2*Xfb + (1-l2)*Xfb[si]},
                l2*yb + (1-l2)*yb[si])

Xf_tr_scaled = scaler.transform(X_flat_base[idx_tr])
train_gen = AugMixupGen(
    X_2d_base[idx_tr], Xf_tr_scaled,
    y_cat[idx_tr], bs=BATCH_SIZE, alpha=0.4, aug_rounds=AUG_ROUNDS
)
print(f'Steps per epoch: {len(train_gen)}  (effective samples: {len(idx_tr)*(AUG_ROUNDS+1)})')

In [ ]:
def se_block(x, r=16):
    c   = x.shape[-1]
    gap = layers.GlobalAveragePooling2D()(x)
    gap = layers.Dense(max(c//r,4), activation='relu')(gap)
    gap = layers.Dense(c, activation='sigmoid')(gap)
    return layers.Multiply()([x, layers.Reshape((1,1,c))(gap)])

def cbam_block(x, r=16):
    x   = se_block(x, r)
    avg = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    mx  = layers.Lambda(lambda t: tf.reduce_max( t, axis=-1, keepdims=True))(x)
    spa = layers.Conv2D(1, 7, padding='same', activation='sigmoid', use_bias=False)(
              layers.Concatenate(axis=-1)([avg, mx]))
    return layers.Multiply()([x, spa])

def res_block(x, filters, drop):
    sc = layers.BatchNormalization()(layers.Conv2D(filters,1,padding='same',use_bias=False)(x))
    h  = layers.ReLU()(layers.BatchNormalization()(
             layers.Conv2D(filters,3,padding='same',kernel_regularizer=REG,use_bias=False)(x)))
    h  = layers.BatchNormalization()(
             layers.Conv2D(filters,3,padding='same',kernel_regularizer=REG,use_bias=False)(h))
    h  = cbam_block(h)
    h  = layers.ReLU()(layers.Add()([h, sc]))
    return layers.Dropout(drop)(layers.MaxPooling2D(2)(h))

def attn_pool(x):
    c      = x.shape[-1]
    x_flat = layers.Reshape((-1, c))(x)
    attn   = layers.Softmax(axis=1)(layers.Dense(1)(x_flat))
    return layers.Lambda(lambda t: tf.reduce_sum(t[0]*t[1], axis=1))([x_flat, attn])

def build_model(flat_dim, spec_shape, nc):
    si = layers.Input(shape=spec_shape, name='spectrogram')
    x  = layers.ReLU()(layers.BatchNormalization()(
             layers.Conv2D(32,3,padding='same',use_bias=False)(si)))
    x  = res_block(x, 64,  0.20)
    x  = res_block(x, 128, 0.25)
    x  = res_block(x, 256, 0.30)
    x  = res_block(x, 512, 0.35)
    x1 = layers.Dropout(0.5)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(256, kernel_regularizer=REG, use_bias=False)(attn_pool(x)))))

    fi = layers.Input(shape=(flat_dim,), name='features')
    f  = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(512, kernel_regularizer=REG, use_bias=False)(fi))))
    f2 = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(256, kernel_regularizer=REG, use_bias=False)(f))))
    f2 = layers.Add()([f2, layers.Dense(256,use_bias=False)(f)])
    f3 = layers.Dropout(0.3)(layers.ReLU()(layers.BatchNormalization()(
             layers.Dense(128, kernel_regularizer=REG, use_bias=False)(f2))))
    x2 = layers.Add()([f3, layers.Dense(128,use_bias=False)(f2)])

    m    = layers.Concatenate()([x1, x2])
    gate = layers.Dense(384, activation='sigmoid', kernel_regularizer=REG)(m)
    m    = layers.Multiply()([m, gate])
    x = layers.Dropout(0.5)(layers.ReLU()(layers.BatchNormalization()(
            layers.Dense(256, kernel_regularizer=REG, use_bias=False)(m))))
    x = layers.Dropout(0.4)(layers.ReLU()(layers.BatchNormalization()(
            layers.Dense(128, kernel_regularizer=REG, use_bias=False)(x))))
    out = layers.Dense(nc, activation='softmax')(x)
    return models.Model([si, fi], out)

model = build_model(Xf_tr_scaled.shape[1], X_2d_base.shape[1:], NC)
model.summary()

In [ ]:
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, smoothing=0.1, **kw):
        super().__init__(**kw)
        self.gamma = gamma; self.smoothing = smoothing
    def call(self, y_true, y_pred):
        nc   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        ys   = y_true*(1-self.smoothing) + self.smoothing/nc
        yp   = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt   = tf.reduce_sum(y_true*yp, axis=-1, keepdims=True)
        return tf.reduce_mean(tf.pow(1-pt, self.gamma) * (-ys*tf.math.log(yp)))
    def get_config(self):
        c = super().get_config(); c.update({'gamma':self.gamma,'smoothing':self.smoothing}); return c

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak, wu_steps, total, min_lr=1e-6):
        super().__init__()
        self.peak=float(peak); self.wu=float(wu_steps)
        self.total=float(total); self.min_lr=float(min_lr)
    def __call__(self, step):
        s   = tf.cast(step, tf.float32)
        wu  = self.peak * s / self.wu
        cos = self.min_lr + (self.peak-self.min_lr)*0.5*(1+tf.cos(np.pi*(s-self.wu)/(self.total-self.wu)))
        return tf.where(s < self.wu, wu, cos)
    def get_config(self):
        return {'peak':self.peak,'wu_steps':self.wu,'total':self.total,'min_lr':self.min_lr}

In [ ]:
spe         = len(train_gen)
total_steps = spe * EPOCHS
wu_steps    = spe * WARMUP_EPOCHS

model.compile(
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate = WarmupCosine(1e-3, wu_steps, total_steps),
        weight_decay  = 1e-4),
    loss    = FocalLoss(gamma=2.0, smoothing=0.1),
    metrics = ['accuracy']
)

# Note: ReduceLROnPlateau removed — conflicts with WarmupCosine schedule
cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(OUTPUT_DIR+'best_ser_v2.keras', monitor='val_accuracy',
                              save_best_only=True, verbose=1),
]

history = model.fit(
    train_gen,
    validation_data = ({'spectrogram': X_va_2, 'features': X_va_f}, y_va),
    epochs          = EPOCHS,
    class_weight    = class_wts,
    callbacks       = cb_list,
    verbose         = 1
)

In [ ]:
def tta_predict(mdl, X2, Xf, n=15):
    preds = [mdl.predict({'spectrogram':X2,'features':Xf},verbose=0)]
    for _ in range(n-1):
        X2a = np.array([spec_augment(X2[j],fm=10,tm=15,n=1)
                        + np.random.randn(*X2[j].shape).astype(np.float32)*0.01
                        for j in range(len(Xf))], dtype=np.float32)
        Xfa = (Xf + np.random.randn(*Xf.shape).astype(np.float32)*0.05)
        preds.append(mdl.predict({'spectrogram':X2a,'features':Xfa},verbose=0))
    return np.mean(preds, axis=0)

# Auto-detect model path — works after training OR when uploaded as a dataset
def _find_model():
    candidates = [
        OUTPUT_DIR + 'best_ser_v2.keras',                          # after training
        '/kaggle/input/ser-saved-model/best_ser_v2.keras',         # uploaded dataset
    ] + glob.glob('/kaggle/input/**/best_ser_v2.keras', recursive=True)
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "best_ser_v2.keras not found. Either run training (cell 12) first, "
        "or upload it via Add Data → ser-saved-model dataset.")

model_path = _find_model()
print(f'Loading model from: {model_path}')
best_model = tf.keras.models.load_model(
    model_path,
    custom_objects={'FocalLoss':FocalLoss,'WarmupCosine':WarmupCosine})

_, acc_std = best_model.evaluate({'spectrogram':X_te_2,'features':X_te_f}, y_te, verbose=0)
print(f'Standard Test Accuracy : {acc_std*100:.2f}%')

tta_p    = tta_predict(best_model, X_te_2, X_te_f, n=15)
y_te_int = np.argmax(y_te,  axis=1)
tta_int  = np.argmax(tta_p, axis=1)
acc_tta  = np.mean(tta_int == y_te_int)
f1_tta   = f1_score(y_te_int, tta_int, average='macro')
print(f'TTA Test Accuracy      : {acc_tta*100:.2f}%')
print(f'TTA Macro F1           : {f1_tta*100:.2f}%')
print()
print(classification_report(y_te_int, tta_int, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')

cm = confusion_matrix(y_te_int, tta_int)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title(f'Confusion Matrix — TTA {acc_tta*100:.1f}%')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR+'results.png', dpi=150)
plt.show()

print(f'\n=== FINAL ===')
print(f'Standard : {acc_std*100:.2f}%')
print(f'TTA      : {acc_tta*100:.2f}%')
print(f'Macro F1 : {f1_tta*100:.2f}%')
print(f'\nModel saved to: {OUTPUT_DIR}best_ser_v2.keras')